In [1]:
import sys

print(sys.executable)

c:\Users\ASUS\.conda\envs\fraud\python.exe


In [2]:
import pandas as pd

data = pd.read_csv("fraud_dataset_new.csv")
data

,Teks,label
0,Penetapan rekanan tanpa proses kompetitif. Kom...,1
1,meeting di hotel. kontak via aplikasi amankomu...,1
2,Penggunaan fasilitas kantor harus sesuai denga...,0
3,Pengajuan cuti tahunan harus dilakukan minimal...,0
4,Tender proyek; pengadaan pemerintahProyek kons...,1
...,...,...
957,Verif cepat diperlukan supaya akun kamu tetap ...,1
958,Audit internal tetap dijalankan sesuai rencana...,0
959,Penggunaan anggaran dilaporkan secara transpar...,0
960,Saldo baru bisa dicairkan setelah semua tahapa...,1


In [3]:
print(data['label'].value_counts())

label
1    558
0    404
Name: count, dtype: int64


In [4]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

In [5]:
import re

# Case folding
def casefolding(text):
    return text.lower()

data['Teks'] = data['Teks'].apply(casefolding)

# Cleansing
def cleansing(text):
    text = re.sub(r'[?|$|.|!_:")(-+,]', '', text) # hapus punctuation
    text = re.sub(r'\d+', '', text) # hapus numbers
    text = re.sub(r'\b[a-zA-Z]\b', '', text) # hapus single characters
    text = re.sub('\s+', ' ', text) # hapus multiple spaces
    return text.strip()

data['Teks'] = data['Teks'].apply(cleansing)

# tokenisasi dan hapus stopwords Sastrawi
def sastrawi_tokenization(text):
    text = stopword_remover.remove(text)
    return text.split()

data['Teks'] = data['Teks'].apply(sastrawi_tokenization)

# lemmatization (Stemming) Sastrawi
def stemming(tokens):
    return ' '.join([stemmer.stem(token) for token in tokens])

data['Teks'] = data['Teks'].apply(stemming)


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

X = data['Teks']
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('knn', KNeighborsClassifier())
])

param_grid = {
    'knn__n_neighbors': [3,5,7,9,11,15],
    'knn__weights': ['uniform','distance'],
    'knn__metric': ['euclidean','manhattan']
}

model = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

model.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'knn__metric': ['euclidean', 'manhattan'], 'knn__n_neighbors': [3, 5, ...], 'knn__weights': ['uniform', 'distance']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candi

In [7]:
best_model = model.best_estimator_
tfidf = best_model.named_steps['tfidf']

print("Best Parameters:", model.best_params_)
print("Jumlah fitur TF-IDF:", len(tfidf.get_feature_names_out()))
print("Jumlah data latih:", X_train.shape[0])
y_pred = model.predict(X_test)

Best Parameters: {'knn__metric': 'euclidean', 'knn__n_neighbors': 3, 'knn__weights': 'uniform'}
Jumlah fitur TF-IDF: 658
Jumlah data latih: 769


In [8]:
from sklearn.metrics import classification_report

print("\nClassification Report:\n", classification_report(y_test, y_pred))


Classification Report:
               precision    recall  f1-score   support

           0       0.94      1.00      0.97        81
           1       1.00      0.96      0.98       112

    accuracy                           0.97       193
   macro avg       0.97      0.98      0.97       193
weighted avg       0.98      0.97      0.97       193



In [9]:
import joblib

joblib.dump(best_model, 'knn_pipeline.pkl')

['knn_pipeline.pkl']

In [10]:
model = joblib.load('knn_pipeline.pkl')

def predict_fraud(text):
    prediction = model.predict([text])

    result = prediction[0]
    if result == 1:
        return f"Prediksi Fraud: [1] (FRAUD DETECTED)"
    else:
        return f"Prediksi Fraud: [0] (NORMAL TEXT)"
    
new_text = "kontak via aplikasi amankomunikasi rahasia"
result = predict_fraud(new_text)
print(result)

Prediksi Fraud: [1] (FRAUD DETECTED)


In [11]:
test_texts = [
"kontak via aplikasi aman komunikasi rahasia",
"Rapat akan dilaksanakan besok pukul 10.00",
"Fee 15% untuk Anda sebagai komisi",
"Laporan keuangan triwulan sudah selesai", 
"Hubungi saya pribadi saja"
]
print("\nUJI BEBERAPA TEKS:")
for text in test_texts:
    result = predict_fraud(text)
    print(f"\nTeks: {text}...")
    print(f"Hasil: {result}")


UJI BEBERAPA TEKS:

Teks: kontak via aplikasi aman komunikasi rahasia...
Hasil: Prediksi Fraud: [1] (FRAUD DETECTED)

Teks: Rapat akan dilaksanakan besok pukul 10.00...
Hasil: Prediksi Fraud: [0] (NORMAL TEXT)

Teks: Fee 15% untuk Anda sebagai komisi...
Hasil: Prediksi Fraud: [1] (FRAUD DETECTED)

Teks: Laporan keuangan triwulan sudah selesai...
Hasil: Prediksi Fraud: [0] (NORMAL TEXT)

Teks: Hubungi saya pribadi saja...
Hasil: Prediksi Fraud: [1] (FRAUD DETECTED)
